# Initialization Scaling
This notebook studies the effect of the scaling of the Tutte embedding on convergence for Symmetric Dirichlet parametrization.

### Minimizing the gradient norm:

When we scale the UV coordinates by a global factor of $s$, the Dirichlet energy gradient ${\bf g}_a$ scales by $s$ and the $F^{-1}$ term of the gradient scales by $1/s^3$. Thus the squared norm of the scaled gradient is:
$$
n(s) = \left \|s {\bf g}_a + \frac{1}{s^3} {\bf g}_b\right\|^2 = s^2 \|{\bf g}_a\|^2 + 2 \frac{1}{s^2} {\bf g}_a \cdot {\bf g}_b + \frac{1}{s^6} \|{\bf g}_b\|^2
$$
We now seek $s$ minimizing this squared norm:
$$
n'(s) = 2 s \|{\bf g}_a\|^2 - 4 \frac{1}{s^3} {\bf g}_a \cdot {\bf g}_b - 6 \frac{1}{s^7} \|{\bf g}_b\|^2.
$$
This derivative vanishes if and only if:
$$
s^8 \|{\bf g}_a\|^2 - 2 s^4 {\bf g}_a \cdot {\bf g}_b - 3 \|{\bf g}_b\|^2 = 0
$$
The change of variables $\tilde s = s^4$ converts this to a quadratic equation:
$$
s^2 \|{\bf g}_a\|^2 - 2 s {\bf g}_a \cdot {\bf g}_b - 3 \tilde \|{\bf g}_b\|^2 = 0
$$
for which the positive solution is:
$$
\tilde s = \frac{{\bf g}_a \cdot {\bf g}_b + \sqrt{({\bf g}_a \cdot {\bf g}_b)^2 + 3 \|{\bf g}_a\|^2\|{\bf g}_b\|^2}}{\|{\bf g}_a\|^2}.
$$

In [ ]:
import os
os.environ['OMP_NUM_THREADS'] = '1'
os.environ['VECLIB_MAXIMUM_THREADS'] = '1'

In [ ]:
import MeshFEM
import mesh, mesh_energy, py_newton_optimizer, viewer
import parametrization, param_utils, benchmark
import energy
import numpy as np

In [ ]:
model = 'lucy.msh.xz'
# model = 'hilbert_curve.msh.xz'
# model = 'bird.msh.xz'
# model = 'hand.msh'
# model = 'cow2Disc.msh'

In [ ]:
m = param_utils.load(f'../models/{model}')

In [ ]:
# Scale so that the surface area is pi (to match [Su et al. 2020])
m.setVertices(m.vertices() * np.sqrt(np.pi / m.volume))

In [ ]:
uv = mesh_energy.NodalVars(m, 2)
uv_init = param_utils.tutteInitialization(m)
uv.setVars(uv_init.ravel())

In [ ]:
e = energy.SymmetricDirichlet(2)
param = mesh_energy.Parametrization(m, uv, e)
objectives = [param]

prob = py_newton_optimizer.NewtonMultiobjectiveProblem(uv, objectives)

In [ ]:
methods = ['orig', 'energy_minimal', 'grad_minimal', 'full_tension', 'psd', 'bulk_tension']
def initialization_scale(method):
    if (method == 'orig'):
        return 1
    if (method == 'energy_minimal'):
        import dirichlet_demo
        a = dirichlet_demo.param_dirichlet_edensity(m, uv).objective()
        b = param.objective() - a
        return (b / a)**(1/4)
    if (method == 'grad_minimal'):
        import dirichlet_demo
        g_a = dirichlet_demo.param_dirichlet_edensity(m, uv).gradient()
        g_b = param.gradient() - g_a

        g_a_dot_g_b = g_a.dot(g_b)
        g_a_sqnorm = g_a.dot(g_a)
        g_b_sqnorm = g_b.dot(g_b)
        stilde = (g_a_dot_g_b + np.sqrt(g_a_dot_g_b ** 2 + 3 * g_a_sqnorm * g_b_sqnorm)) / g_a_sqnorm
        return stilde**(1/4)
    if (method == 'full_tension'):
        return 1 / np.min([np.linalg.svd(param.elementJacobian(i), compute_uv=False).min() for i in range(param.numElements())])
    if (method == 'psd'):
        # Solve for the scale factor that makes all per-element Symmetric Dirichlet Hessians PSD
        s = 1
        for i in range(param.numElements()):
            F = param.elementJacobian(i)
            I3 = np.linalg.det(F) # scales like s^2
            I2 = F.ravel().dot(F.ravel()) # scales like s^2
            I3Sq = I3 * I3
            I3Cu = I3Sq * I3 # scales like s^6
            a = 1.0/I3Sq - I2/I3Cu # scales like 1 / s^4
            # lambda_4 = 1 + a / s^4
            if a < 0: s = max(s, (-a)**(1/4))
        return s
    if (method == 'bulk_tension'):
        # Solve for the scale factor that makes all determinants greater than 1
        s = 1 / np.sqrt(np.min([np.linalg.det(param.elementJacobian(i)) for i in range(param.numElements())]))
        return s
    raise Exception('Unknown method')

In [ ]:
def run_with_method(method):
    uv.setVars(uv_init.ravel())
    uv.setVars(initialization_scale(method) * uv_init.ravel())

    import flip_avoiding_step_length
    prob.initialFeasibleStepLengthComputer = flip_avoiding_step_length.FlipAvoidingStepLength(m.elements())
    prob.initialFeasibleStepLengthComputer.backoffFactor = 0.9

    # Work around energy nullspace by adding a small shift
    prob.hessianShift = 1e-12
    prob.useRelativeHessianShift = True
    param.useXBasedProjection = False
    opt = prob.optimizer()
    opt.options.niter = 500
    opt.options.gradTol = 0.1

    opt.options.hessianProjectionController.numConsecutiveIndefiniteStepsBeforeEnable = 0
    opt.options.hessianProjectionController.numProjectionStepsBeforeDisable = 2
    opt.options.hessianProjectionController.startWithProjectionActive = False

    benchmark.reset()
    rep = opt.optimize()
    benchmark.report()
    return rep

In [ ]:
results = [run_with_method(me) for me in methods]

In [ ]:
from matplotlib import pyplot as plt

In [ ]:
plt.figure(figsize=(8, 4))
plt.subplot(1, 2, 1)
plt.suptitle(f'Initialization Scaling Experiment - {model}')
for n, r in zip(methods, results): plt.semilogy(r.energy, label=n)
plt.xlabel('iteration')
plt.ylabel('energy')

plt.subplot(1, 2, 2)
for n, r in zip(methods, results): plt.semilogy(r.freeGradientNorm, label=n)
plt.xlabel('iteration')
plt.ylabel('gradient norm')

plt.legend()
plt.tight_layout()